In [1]:
import re
import numpy as np
import pandas as pd
import anndata as ad

In [2]:
depmap = pd.read_csv("/cluster/work/boeva/eheiss/datasets/DepMap/CRISPRGeneEffect.csv")

In [3]:
genes = pd.read_csv("/cluster/work/boeva/eheiss/datasets/DepMap/Gene.csv", low_memory = False)

In [4]:
def depmap_to_anndata(depmap: pd.DataFrame, genes: pd.DataFrame) -> ad.AnnData:
    depmap = depmap.copy()
    genes = genes.copy()

    # --- 1) get model IDs from first column ---
    first_col = depmap.columns[0]
    model_ids = depmap[first_col].astype(str).values

    # remaining columns are genes like "A1BG (1)"
    gene_cols = depmap.columns[1:]

    # --- 2) parse DepMap gene headers into symbol + entrez_id ---
    parsed = []
    pat = re.compile(r"^(.*?)\s*\((\d+)\)$")

    for col in gene_cols:
        col = str(col)
        m = pat.match(col)
        if m:
            symbol = m.group(1).strip()
            entrez_id = m.group(2).strip()
        else:
            symbol = col.strip()
            entrez_id = pd.NA
        parsed.append((col, symbol, entrez_id))

    parsed_df = pd.DataFrame(parsed, columns=["depmap_col", "symbol", "entrez_id"])

    # --- 3) prepare Gene.csv mapping ---
    genes["symbol"] = genes["symbol"].astype(str).str.strip()
    genes["entrez_id"] = genes["entrez_id"].astype("string").str.strip()
    genes["ensembl_gene_id"] = genes["ensembl_gene_id"].astype("string").str.strip()

    # first try exact match on symbol + entrez_id
    map_df = parsed_df.merge(
        genes[["symbol", "entrez_id", "ensembl_gene_id", "name"]].drop_duplicates(),
        on=["symbol", "entrez_id"],
        how="left"
    )

    # fallback: symbol-only for those still unmapped
    missing = map_df["ensembl_gene_id"].isna()
    if missing.any():
        symbol_only = (
            genes[["symbol", "ensembl_gene_id", "name"]]
            .dropna(subset=["ensembl_gene_id"])
            .drop_duplicates(subset=["symbol"])
        )
        fallback = parsed_df.loc[missing, ["depmap_col", "symbol"]].merge(
            symbol_only,
            on="symbol",
            how="left"
        )
        map_df.loc[missing, "ensembl_gene_id"] = fallback["ensembl_gene_id"].values
        map_df.loc[missing, "name"] = fallback["name"].values

    # keep only mapped genes
    map_df = map_df.dropna(subset=["ensembl_gene_id"]).copy()

    # strip Ensembl version just in case
    map_df["ensembl_gene_id"] = map_df["ensembl_gene_id"].str.split(".").str[0]

    # --- 4) subset expression matrix to mapped genes ---
    X_df = depmap.loc[:, map_df["depmap_col"]].copy()
    X_df.index = model_ids
    X_df.columns = map_df["ensembl_gene_id"].values

    # numeric
    X_df = X_df.apply(pd.to_numeric, errors="coerce").fillna(0.0)

    # --- 5) collapse duplicate Ensembl IDs after mapping ---
    X_df = X_df.T.groupby(level=0).mean().T

    # gene metadata
    var = (
        map_df[["ensembl_gene_id", "symbol", "name"]]
        .drop_duplicates(subset=["ensembl_gene_id"])
        .set_index("ensembl_gene_id")
        .reindex(X_df.columns)
    )

    obs = pd.DataFrame(index=X_df.index)
    obs.index.name = "model_id"

    adata = ad.AnnData(
        X=X_df.to_numpy(dtype=np.float32),
        obs=obs,
        var=var
    )

    return adata

In [5]:
adata = depmap_to_anndata(depmap, genes)
adata

/scratch/slurm-job.8999018/ipykernel_1328695/2737600778.py:54: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  map_df.loc[missing, "ensembl_gene_id"] = fallback["ensembl_gene_id"].values


AnnData object with n_obs × n_vars = 1208 × 18531
    var: 'symbol', 'name'

In [6]:
adata.write("/cluster/work/boeva/eheiss/datasets/DepMap/depmap.h5ad")

In [7]:
expr = pd.read_csv(
    "/cluster/work/boeva/eheiss/datasets/DepMap/OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv",
    index_col=0,
)

# One row per cell line (drop duplicate sequencing runs)
expr = expr[expr["IsDefaultEntryForModel"] == "Yes"].copy()

# Drop metadata columns, keep ModelID as the row identifier
meta_cols = ["SequencingID", "ModelConditionID", "IsDefaultEntryForModel", "IsDefaultEntryForMC"]
gene_cols = [c for c in expr.columns if c not in meta_cols and c != "ModelID"]
expr_clean = expr[["ModelID"] + gene_cols].reset_index(drop=True)

# Reverse log1p to get back to TPM, then convert to AnnData
gene_matrix = expr_clean[gene_cols].to_numpy(dtype=np.float32)
gene_matrix = np.expm1(gene_matrix)  # TPM = exp(log1p(TPM)) - 1
expr_clean[gene_cols] = gene_matrix

# Reuse the existing function — same SYMBOL (EntrezID) column format
adata_expr = depmap_to_anndata(expr_clean, genes)
adata_expr


/scratch/slurm-job.8999018/ipykernel_1328695/2737600778.py:54: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  map_df.loc[missing, "ensembl_gene_id"] = fallback["ensembl_gene_id"].values


AnnData object with n_obs × n_vars = 1719 × 19215
    var: 'symbol', 'name'

In [8]:
adata_expr.write("/cluster/work/boeva/eheiss/datasets/DepMap/expression.h5ad")
